# 이용빈도 (Frequency) 모델

# 신용카드 이용빈도 기반 활동성 저하 및 이탈 예측
> **목적**: 결제 건수와 활동 일수의 감소를 추적하여 우리 카드가 고객의 일상에서 '주카드' 지위를 상실해가는 과정을 통계적으로 규명

### 주요 분석 스토리라인:
1. **가설 설정**: 이탈 고객은 금액의 급격한 변화보다 결제 횟수와 방문 주기(활동 밀도)가 먼저 불규칙해질 것이라 가정
2. **피처 엔지니어링**: 이용건수 변화율, 활동밀도, 일평균 결제건수 등 활동성 중심 파생 변수 **산출**
3. **전략적 비교 분석**: 
   - 이용강도(F1)와 이용빈도(F2) 모델의 이탈자 적중 특성 차이 분석 및 상호 보완성 **확인**
   - 일상 결제 패턴 복원을 위해 대규모 트렌드 처리에 능한 **LightGBM** 기반 고도화 모델 **구축**
4. **통계적 검증**: 리텐션 그룹 간 활동 지표의 유의미한 차이를 **Mann-Whitney U Test**를 통해 **증명**
5. **인사이트 도출**: 활동성 저하 세그먼트 정의 및 타겟 마케팅용 예측 데이터 **생성**

In [ ]:
# Step 0. 라이브러리 및 한글 폰트 설정

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import gc
import polars as pl
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    classification_report, f1_score, precision_recall_curve, 
    auc, recall_score, precision_score, average_precision_score,
    calibration_curve, confusion_matrix
)
import shap

# 한글 폰트 설정 (Windows: Malgun Gothic)
plt.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False

print("STEP 0: 환경 설정 및 라이브러리 로드 완료")

In [ ]:
# STEP 1. 환경 설정 및 데이터 로드

# 통일된 마스터 데이터셋(final_master_dataset.parquet)을 로드합니다.
df = pl.read_parquet('final_master_dataset.parquet').to_pandas()

print(f"STEP 1: 데이터 로드 완료. Shape: {df.shape}")
# 이용빈도 모델의 베이스라인 이탈률 확인
print(f"전체 이탈률(이용건수 기준): {df['is_churn'].mean():.2%}")

In [ ]:
# Step 2. Y값 정의 및 변수 및 파생변수 생성

def engineer_frequency_features(df):
    print("STEP 2: 이용빈도(활동성) 중심 파생변수 생성 중...")
    eps = 1e-9
    
    # 1. 이용건수 추세 (Frequency Trend)
    df['이용건수_기울기'] = df['이용건수_신용_B0M'] - df['이용건수_신용_B1M']
    df['이용건수_변화율'] = df['이용건수_기울기'] / (df['이용건수_신용_B1M'] + eps)
    
    # 2. 활동 밀도 (Activity Intensity)
    # 한 달 중 며칠이나 카드를 사용하는가?
    df['활동밀도'] = df['활동일수_B0M'] / 31 
    df['일평균_결제건수'] = df['이용건수_신용_B0M'] / (df['활동일수_B0M'] + eps)
    
    # 3. 주카드 지위 지표
    # 전체 이용건수 중 특정 업종이나 온라인 비중이 어떻게 변하는가?
    df['건수기준_온라인비중'] = df['이용건수_온라인_B0M'] / (df['이용건수_신용_B0M'] + eps)
    
    # 4. 장기 빈도 추세 (3개월 평균 건수)
    df['이용건수_3M_Avg'] = df[['이용건수_신용_B0M', '이용건수_신용_B1M', '이용건수_신용_B2M']].mean(axis=1)
    
    return df

df = engineer_frequency_features(df)

In [ ]:
# Step 3. 베이스모델(RF) 선정 및 모델링 (타임스플릿 적용)

print("STEP 3: Random Forest 베이스라인 학습 (TimeSeriesSplit 적용)...")

tscv = TimeSeriesSplit(n_splits=3)

# 이용빈도 핵심 변수 리스트 확정
frequency_features = [
    '이용건수_신용_B0M', '이용건수_신용_B1M', '이용건수_변화율', 
    '이용건수_기울기', '활동밀도', '일평균_결제건수', 
    '이용건수_3M_Avg', '건수기준_온라인비중', '활동일수_B0M'
]

X = df[frequency_features]
y = df['is_churn']

rf_base = RandomForestClassifier(
    n_estimators=200, 
    max_depth=10, 
    class_weight='balanced', 
    random_state=42,
    n_jobs=-1
)

for i, (train_idx, val_idx) in enumerate(tscv.split(X)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    rf_base.fit(X_train, y_train)
    print(f"Fold {i+1} 학습 완료")

In [ ]:
# Step 4. 피처 목록 확정 (SHAP, Permutation Importance, TopK Lift)

print("STEP 4: 활동성 지표 유효성 분석 (SHAP & Lift)...")

# SHAP 분석: 활동일수와 이용건수 변화율의 영향력 확인
explainer = shap.TreeExplainer(rf_base)
shap_values = explainer.shap_values(X.sample(1000))
shap.summary_plot(shap_values, X.sample(1000))

# Top-K Lift 산출
freq_probs = rf_base.predict_proba(X)[:, 1]
lift_df = pd.DataFrame({'prob': freq_probs, 'actual': y}).sort_values('prob', ascending=False)
top_10_actual = lift_df.head(int(len(lift_df)*0.1))['actual'].mean()
print(f"Frequency Model Top 10% Lift: {top_10_actual / y.mean():.2f}배")

In [ ]:
# Step 5. 모델 고도화 위한 스태킹 진행 (RF, LGBM, CAT, XGB)

print("STEP 5: 앙상블 스태킹(Stacking) 수행...")

stack_models = {
    'RF': rf_base,
    'XGB': XGBClassifier(n_estimators=200, scale_pos_weight=5, random_state=42),
    'LGBM': LGBMClassifier(n_estimators=200, scale_pos_weight=5, random_state=42),
    'CAT': CatBoostClassifier(iterations=200, auto_class_weights='Balanced', verbose=0)
}

oof_preds = pd.DataFrame(index=df.index)
for name, model in stack_models.items():
    model.fit(X, y)
    oof_preds[name] = model.predict_proba(X)[:, 1]

# Meta-Learner: ElasticNet
meta_learner = LogisticRegression(penalty='elasticnet', solver='saga', l1_ratio=0.5)
meta_learner.fit(oof_preds, y)

In [ ]:
# Step 6. 주제별 엔진 선정 (이용빈도 - LightGBM 선정)

print("STEP 6: 이용빈도 최종 엔진(LightGBM) 모델링...")

final_lgbm = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    scale_pos_weight=5,
    random_state=42
)

final_lgbm.fit(X, y)
final_frequency_probs = final_lgbm.predict_proba(X)[:, 1]

In [ ]:
# Step 7. 2019년 1월 이탈자 예측 결과 저장

print("STEP 7: 마케팅 통합을 위한 예측 결과 저장 (Frequency)...")

result_df = df[['발급회원번호', 'is_churn']].copy()
result_df['frequency_prob'] = final_frequency_probs

# 예측 결과 저장 (marketing_strategy.py에서 병합 대상)
result_df.to_csv('frequency_churn_full_results.csv', index=False, encoding='utf-8-sig')
print("frequency_churn_full_results.csv 저장 완료.")

In [ ]:
# Step 8. 모델 검증 (PSI, Calibration Curve)

print("STEP 8: 최종 모델 진단 (Calibration Curve)...")

prob_true, prob_pred = calibration_curve(y, final_frequency_probs, n_bins=10)
plt.figure(figsize=(8, 6))
plt.plot(prob_pred, prob_true, marker='s', color='green', label='Frequency LGBM')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.title('Frequency Model Reliability (Calibration)')
plt.xlabel('Predicted Probability')
plt.ylabel('Actual Churn Rate')
plt.legend()
plt.show()

# PSI 산출 로직 (전처리 및 모델링 단계별 데이터 분포 확인)
# psi_val = calculate_psi(expected, actual)

In [ ]:
# Step 9. 모델 검증 PR Curve

# PR Curve 시각화
precision, recall, _ = precision_recall_curve(y, final_frequency_probs)
pr_auc = auc(recall, precision)
plt.figure(figsize=(8, 6))
plt.plot(recall, precision, color='darkorange', label=f'PR-AUC = {pr_auc:.4f}')
plt.title('Frequency Model PR-Curve')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.legend()
plt.show()